# MoA — TabPFN (Basic preprocessing + PCA 500 · drug_id GroupKFold)

`tabpfn_basic_pca500_holdout.ipynb` 와 동일한 파이프라인 + **Holdout split 을 drug_id 기준 GroupKFold 로 변경**.

목적:
- 같은 약물의 샘플이 train/val 에 동시에 들어가는 data leakage 제거
- drug 수준의 일반화 성능을 측정

전처리 흐름 (원 코드 미러):
1. g-/c- 컬럼별 QuantileTransformer — train_split fit → val transform
2. `cp_type==ctl_vehicle` 제거 후 **GroupKFold (groups=drug_id) split**
3. cp_time/cp_dose 정수 인코딩
4. **PCA(n=500) — train_split 에 fit → 양쪽 transform**
5. 206개 target 각각에 TabPFNClassifier 독립 학습

## 1. 경로 / 설정

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
TABPFN_TOKEN = os.getenv("TABPFN_TOKEN")

In [2]:
DATA_DIR   = './'
OUTPUT_DIR = './moa_outputs_tabpfn_drug_id_split'

DEVICE     = 'cpu'
SEED       = 42
N_SPLITS   = 5
VAL_FOLD   = 0

PCA_N_COMPONENTS = 500   # TabPFN v2 pretraining 피처 한계에 맞춤. 실험용 변수.
MAX_TARGETS      = 3  # 디버그용. 전체는 None.

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [9]:
# !pip install tabpfn iterative-stratification

## 2. 데이터 로드 (train 만)

In [3]:
import numpy as np
import pandas as pd
import random, time, warnings
warnings.filterwarnings('ignore')

random.seed(SEED); np.random.seed(SEED)

train_features       = pd.read_csv(f'{DATA_DIR}/train_features.csv')
train_targets_scored = pd.read_csv(f'{DATA_DIR}/train_targets_scored.csv')
train_drug           = pd.read_csv('../data/raw/train_drug.csv')

print('train_features      :', train_features.shape)
print('train_targets_scored:', train_targets_scored.shape)
print('train_drug          :', train_drug.shape)

train_features      : (23814, 876)
train_targets_scored: (23814, 207)
train_drug          : (23814, 2)


## 3. Holdout split — ctl_vehicle 제거 후 GroupKFold (drug_id 기준)

In [4]:
from sklearn.model_selection import GroupKFold

target_cols = [c for c in train_targets_scored.columns if c != 'sig_id']  # 206

df_full = train_features.merge(train_targets_scored, on='sig_id')
df_full = df_full.merge(train_drug, on='sig_id', how='left')
df_full = df_full[df_full['cp_type'] != 'ctl_vehicle'].reset_index(drop=True)

gkf = GroupKFold(n_splits=N_SPLITS)
groups = df_full['drug_id'].values
splits = list(gkf.split(X=df_full, y=df_full[target_cols].values, groups=groups))
tr_idx, va_idx = splits[VAL_FOLD]

train_split = df_full.loc[tr_idx].reset_index(drop=True)
val_split   = df_full.loc[va_idx].reset_index(drop=True)
print(f'train_split: {train_split.shape}   val_split: {val_split.shape}')
assert len(set(train_split.drug_id) & set(val_split.drug_id)) == 0, 'drug leakage!'

train_split: (17558, 1083)   val_split: (4390, 1083)


## 4. 전처리 — g-/c- 컬럼별 QuantileTransformer (원 코드 미러)

In [ ]:
from sklearn.preprocessing import QuantileTransformer

GENES = [c for c in train_features.columns if c.startswith('g-')]
CELLS = [c for c in train_features.columns if c.startswith('c-')]
print(f'GENES={len(GENES)}  CELLS={len(CELLS)}')

for col in GENES + CELLS:
    qt = QuantileTransformer(n_quantiles=100, random_state=0, output_distribution='normal')
    train_split[col] = qt.fit_transform(train_split[[col]]).ravel()
    val_split[col]   = qt.transform(val_split[[col]]).ravel()

In [ ]:
# cp_time / cp_dose 인코딩
CP_TIME_MAP = {24:0, 48:1, 72:2}
CP_DOSE_MAP = {'D1':0, 'D2':1}
for df in (train_split, val_split):
    df['cp_time'] = df['cp_time'].map(CP_TIME_MAP)
    df['cp_dose'] = df['cp_dose'].map(CP_DOSE_MAP)

feature_cols = ['cp_time', 'cp_dose'] + GENES + CELLS  # 2 + 772 + 100 = 874

X_train_raw = train_split[feature_cols].astype(np.float32).values
Y_train     = train_split[target_cols].astype(np.float32).values
X_val_raw   = val_split[feature_cols].astype(np.float32).values
Y_val       = val_split[target_cols].astype(np.float32).values

print('X_train_raw:', X_train_raw.shape, ' Y_train:', Y_train.shape)
print('X_val_raw  :', X_val_raw.shape,   ' Y_val  :', Y_val.shape)

## 5. 모델 투입 직전 PCA — train_split 에 fit → 양쪽 transform

In [ ]:
from sklearn.decomposition import PCA

n_comp = min(PCA_N_COMPONENTS, X_train_raw.shape[1], X_train_raw.shape[0])
pca = PCA(n_components=n_comp, random_state=SEED)
pca.fit(X_train_raw)

X_train = pca.transform(X_train_raw).astype(np.float32)
X_val   = pca.transform(X_val_raw).astype(np.float32)

print(f'PCA n_components={n_comp}  explained_variance_ratio sum={pca.explained_variance_ratio_.sum():.4f}')
print('X_train:', X_train.shape, ' X_val:', X_val.shape)

## 6. TabPFN — 206개 binary classifier 루프

In [ ]:
from tabpfn import TabPFNClassifier

# 피처 수는 PCA 로 500 이하로 줄였지만, 샘플 수가 여전히 10k 를 초과 (~17k) → ignore_pretraining_limits 유지
def make_tabpfn():
    return TabPFNClassifier(
        device=DEVICE,
        random_state=SEED,
        ignore_pretraining_limits=True,
    )

targets_to_run = target_cols if MAX_TARGETS is None else target_cols[:MAX_TARGETS]
print(f'Running TabPFN on {len(targets_to_run)} targets …')

In [ ]:
pred_matrix = np.zeros((X_val.shape[0], len(target_cols)), dtype=np.float32)
meta = []
t0 = time.time()

for i, tgt in enumerate(targets_to_run):
    j = target_cols.index(tgt)
    y = Y_train[:, j].astype(int)
    n_pos = int(y.sum())

    if n_pos == 0:
        pred_matrix[:, j] = 0.0
        meta.append((tgt, n_pos, 0.0, 0.0))
        continue

    ts = time.time()
    clf = make_tabpfn()
    clf.fit(X_train, y)
    proba = clf.predict_proba(X_val)
    pos_idx = list(clf.classes_).index(1)
    pred_matrix[:, j] = proba[:, pos_idx].astype(np.float32)
    dt = time.time() - ts
    meta.append((tgt, n_pos, float(pred_matrix[:, j].mean()), dt))

    if (i+1) % 10 == 0 or i == 0:
        elapsed = time.time() - t0
        eta = elapsed / (i+1) * (len(targets_to_run) - (i+1))
        print(f'[{i+1:3d}/{len(targets_to_run)}] {tgt:40s} n_pos={n_pos:4d} dt={dt:5.1f}s  elapsed={elapsed/60:5.1f}m  ETA={eta/60:5.1f}m')

print(f'total: {(time.time()-t0)/60:.1f} min')

## 7. Val 예측 저장

In [ ]:
val_pred_df = pd.DataFrame(pred_matrix, columns=target_cols)
val_pred_df.insert(0, 'sig_id', val_split['sig_id'].values)
pred_path = f'{OUTPUT_DIR}/val_pred_tabpfn_drug_id_split_pca{n_comp}.csv'
val_pred_df.to_csv(pred_path, index=False)
print('wrote', pred_path)
val_pred_df.head()

## 8. 평가 — mean column-wise log-loss

In [ ]:
y_true_arr = Y_val.astype(np.float32)
y_pred_arr = np.clip(pred_matrix.astype(np.float32), 1e-15, 1 - 1e-15)

loss = -(y_true_arr * np.log(y_pred_arr) + (1 - y_true_arr) * np.log(1 - y_pred_arr)).mean()
print(f'Mean column-wise log-loss (TabPFN drug_id split + PCA{n_comp}, fold={VAL_FOLD}): {loss:.5f}')

meta_df = pd.DataFrame(meta, columns=['target', 'n_pos', 'pred_mean', 'fit_predict_sec'])
meta_df.to_csv(f'{OUTPUT_DIR}/tabpfn_drug_id_split_pca{n_comp}_per_target.csv', index=False)
meta_df.head()